In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# تقنيات تخفيف الأخطاء وقمعها

> **Note:** الإصدار التجريبي من نموذج تنفيذ جديد متاح الآن. يوفر نموذج التنفيذ الموجَّه مرونة أكبر عند تخصيص سير عمل تخفيف الأخطاء لديك. راجع دليل [نموذج التنفيذ الموجَّه](/guides/directed-execution-model) لمزيد من المعلومات.


<details>
<summary><b>إصدارات الحزم</b></summary>

تم تطوير الكود في هذه الصفحة باستخدام المتطلبات التالية.
نوصي باستخدام هذه الإصدارات أو ما هو أحدث منها.

```
qiskit-ibm-runtime~=0.43.1
```
</details>
تُستخدَم تقنيات تخفيف الأخطاء وقمعها لتحسين جودة النتائج عند التوسع نحو أحمال عمل أكبر. تقدم هذه الصفحة شرحًا عامًا لتقنيات قمع الأخطاء وتخفيفها المتاحة عبر Qiskit Runtime.

تستورد الخلية التالية الـ primitive الخاصة بـ Estimator وتنشئ backend سيُستخدَم لتهيئة الـ Estimator في خلايا الكود اللاحقة.

In [1]:
from qiskit_ibm_runtime import EstimatorV2 as Estimator
from qiskit_ibm_runtime import QiskitRuntimeService

service = QiskitRuntimeService()
backend = service.least_busy()

## الفصل الديناميكي
تُنفَّذ الدوائر الكمومية على أجهزة IBM&reg; بوصفها تسلسلات من نبضات الميكروويف التي يجب جدولتها وتشغيلها في فترات زمنية دقيقة.
للأسف، يمكن أن تؤدي التفاعلات غير المرغوب فيها بين الـ Qubits إلى أخطاء متماسكة على الـ Qubits الخاملة. يعمل الفصل الديناميكي عن طريق إدراج تسلسلات نبضية على الـ Qubits الخاملة لإلغاء تأثير هذه الأخطاء تقريبًا. كل تسلسل نبضات مُدرَج يعادل عملية هوية، لكن الوجود الفيزيائي للنبضات يؤدي إلى قمع الأخطاء.
هناك الكثير من الاختيارات الممكنة لتسلسلات النبضات، وتحديد أيها أفضل لكل حالة بعينها لا يزال [مجالًا بحثيًا نشطًا](https://journals.aps.org/prapplied/abstract/10.1103/PhysRevApplied.20.064027).

تجدر الإشارة إلى أن الفصل الديناميكي مفيد بصفة رئيسية للدوائر التي تحتوي على فجوات تجلس فيها بعض الـ Qubits خاملةً دون أي عمليات تعمل عليها. إذا كانت العمليات في الدائرة مكتظة بكثافة شديدة بحيث تكون جميع الـ Qubits مشغولة معظم الوقت، فقد لا يُحسِّن إضافة نبضات الفصل الديناميكي الأداء. بل يمكن أن تُسوِّئه في الواقع بسبب الإخفاقات في النبضات نفسها.

يُصوِّر الرسم البياني أدناه الفصل الديناميكي بتسلسل نبضات XX. الدائرة المجردة على اليسار تُعيَّن على جدول زمني لنبضة ميكروويف في أعلى اليمين. يصوّر أسفل اليمين نفس الجدول الزمني، لكن مع إدراج تسلسل من نبضتي X خلال فترة خمول الـ Qubit الأول.

![توضيح الفصل الديناميكي](../docs/images/guides/error-mitigation-explanation/dd.avif)

يمكن تفعيل الفصل الديناميكي عن طريق ضبط `enable` على `True` في [خيارات الفصل الديناميكي](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-dynamical-decoupling-options). يمكن استخدام خيار `sequence_type` للاختيار من بين عدة تسلسلات نبضات مختلفة. نوع التسلسل الافتراضي هو `"XX"`.

تُظهر خلية الكود التالية كيفية تفعيل الفصل الديناميكي للـ Estimator واختيار تسلسل الفصل الديناميكي.

In [2]:
estimator = Estimator(mode=backend)
estimator.options.dynamical_decoupling.enable = True
estimator.options.dynamical_decoupling.sequence_type = "XpXm"

## تلوية باولي
التلوية، المعروفة أيضًا بـ [الترجمة العشوائية](https://journals.aps.org/pra/abstract/10.1103/PhysRevA.94.052325)، هي تقنية شائعة الاستخدام لتحويل قنوات الضوضاء الاعتباطية إلى قنوات ضوضاء ذات هيكل أكثر تحديدًا.

تلوية باولي هي نوع خاص من التلوية يستخدم عمليات باولي. ولها تأثير تحويل أي قناة كمومية إلى قناة باولي. عند تطبيقها منفردةً، يمكنها تخفيف الضوضاء المتماسكة لأن الضوضاء المتماسكة تتراكم تربيعيًا مع عدد العمليات، بينما تتراكم ضوضاء باولي بصورة خطية. كثيرًا ما تُدمَج تلوية باولي مع تقنيات أخرى لتخفيف الأخطاء تعمل بصورة أفضل مع ضوضاء باولي مقارنةً بالضوضاء الاعتباطية.

تُنفَّذ تلوية باولي عن طريق تغليف مجموعة مختارة من البوابات ببوابات باولي أحادية الـ Qubit مختارة عشوائيًا بطريقة تبقى معها التأثير المثالي للبوابة كما هو. والنتيجة أن دائرة واحدة تُستبدَل بمجموعة عشوائية من الدوائر، جميعها بالتأثير المثالي ذاته. عند أخذ العينات من الدائرة، تُستخلَص العينات من حالات عشوائية متعددة بدلًا من حالة واحدة فقط.

![توضيح تلوية باولي](../docs/images/guides/error-mitigation-explanation/pauli_twirling.avif)

بما أن معظم الأخطاء في الأجهزة الكمومية الحالية تأتي من بوابات ثنائية الـ Qubit، فإن هذه التقنية كثيرًا ما تُطبَّق حصرًا على البوابات الثنائية (الأصلية). يصور الرسم البياني التالي بعض تلويات باولي لبوابتَي CNOT وECR. كل دائرة ضمن صف واحد لها التأثير المثالي نفسه.

![توضيح تلويات البوابات](../docs/images/guides/error-mitigation-explanation/gate_twirls.avif)

يمكن تفعيل تلوية باولي عن طريق ضبط `enable_gates` على `True` في [خيارات التلوية](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-twirling-options). تشمل الخيارات الأخرى البارزة:

- `num_randomizations`: عدد نسخ الدوائر التي يُسحَب منها من مجموعة الدوائر الملوَّاة.
- `shots_per_randomization`: عدد اللقطات التي تُؤخَذ من كل نسخة دائرة.

تُظهر خلية الكود التالية كيفية تفعيل تلوية باولي وضبط هذه الخيارات للـ Estimator. لا يُشترَط تحديد أيٍّ من هذه الخيارات صراحةً.

In [3]:
estimator = Estimator(mode=backend)
estimator.options.twirling.enable_gates = True
estimator.options.twirling.num_randomizations = 32
estimator.options.twirling.shots_per_randomization = 100

## إخماد أخطاء القراءة الملوَّاة (TREX)
[إخماد أخطاء القراءة الملوَّاة (TREX)](https://journals.aps.org/pra/abstract/10.1103/PhysRevA.105.032620) يخفف من تأثير أخطاء القياس عند تقدير قيم التوقع لمراقِبات باولي.
يستند إلى مفهوم القياسات الملوَّاة، التي تتحقق عن طريق الاستبدال العشوائي لبوابات القياس بتسلسل من: (1) بوابة باولي X، (2) قياس، و(3) قلب بت كلاسيكي. تمامًا كما في تلوية البوابات القياسية، هذا التسلسل مكافئ لقياس عادي في غياب الضوضاء، كما هو موضح في الرسم البياني التالي:

![توضيح تلوية القياس](../docs/images/guides/error-mitigation-explanation/measurement_twirling.avif)

في حضور خطأ القراءة، يُقلِّب تلوية القياس مصفوفة نقل خطأ القراءة، مما يسهّل عكسها. يتطلب تقدير مصفوفة نقل خطأ القراءة تنفيذ دوائر معايرة إضافية، مما يُضيف عبئًا صغيرًا.

يمكن تفعيل TREX عن طريق ضبط `measure_mitigation` على `True` في [خيارات مرونة Qiskit Runtime](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-resilience-options-v2) للـ Estimator. خيارات تعلم ضوضاء القياس موصوفة [هنا](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-measure-noise-learning-options). كما هو الحال مع تلوية البوابات، يمكنك ضبط عدد التلويات العشوائية للدائرة وعدد اللقطات لكل تلوية.

تُظهر خلية الكود التالية كيفية تفعيل TREX وضبط هذه الخيارات للـ Estimator. لا يُشترَط تحديد أيٍّ من هذه الخيارات صراحةً.

In [4]:
estimator = Estimator(mode=backend)
estimator.options.resilience.measure_mitigation = True
estimator.options.resilience.measure_noise_learning.num_randomizations = 32
estimator.options.resilience.measure_noise_learning.shots_per_randomization = 100

<span id="zne"></span>
## الاستقراء نحو الضوضاء الصفرية (ZNE)
الاستقراء نحو الضوضاء الصفرية (ZNE) هو تقنية لتخفيف الأخطاء في تقدير قيم التوقع للمراقِبات. وإن كانت كثيرًا ما تُحسِّن النتائج، إلا أنها ليست مضمونة لإنتاج نتيجة غير منحازة.

يتكون ZNE من مرحلتين:

1. _تضخيم الضوضاء_: تُنفَّذ الدائرة الكمومية الأصلية عدة مرات عند معدلات ضوضاء مختلفة.
2. _الاستقراء_: يُقدَّر النتيجة المثالية عن طريق الاستقراء من نتائج قيم التوقع المشوشة نحو حد الضوضاء الصفرية.

يمكن تنفيذ كلتا مرحلتي تضخيم الضوضاء والاستقراء بطرق مختلفة متعددة. ينفذ Qiskit Runtime تضخيم الضوضاء عن طريق "طي البوابات الرقمي"، مما يعني أن البوابات ثنائية الـ Qubit تُستبدَل بتسلسلات مكافئة من البوابة ومعكوسها. على سبيل المثال، استبدال وحدوية $U$ بـ $U U^\dagger U$ يعطي عامل تضخيم ضوضاء مقداره 3. بالنسبة للاستقراء، يمكنك الاختيار من بين عدة أشكال دالية، منها ملاءمة خطية أو ملاءمة أسية.
تُوضح الصورة أدناه طي البوابات الرقمي على اليسار، وإجراء الاستقراء على اليمين.

![توضيح ZNE](../docs/images/guides/error-mitigation-explanation/zne.avif)

يمكن تفعيل ZNE عن طريق ضبط `zne_mitigation` على `True` في [خيارات مرونة Qiskit Runtime](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-resilience-options-v2) للـ Estimator.
خيارات Qiskit Runtime الخاصة بـ ZNE موصوفة [هنا](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-zne-options). الخيارات التالية جديرة بالملاحظة:

- `noise_factors`: عوامل الضوضاء المستخدمة لتضخيم الضوضاء.
- `extrapolator`: الشكل الدالي المستخدم للاستقراء.

تُظهر خلية الكود التالية كيفية تفعيل ZNE وضبط هذه الخيارات للـ Estimator. لا يُشترَط تحديد أيٍّ من هذه الخيارات صراحةً.

In [5]:
estimator = Estimator(mode=backend)
estimator.options.resilience.zne_mitigation = True
estimator.options.resilience.zne.noise_factors = (1, 3, 5)
estimator.options.resilience.zne.extrapolator = "exponential"

<span id="pea"></span>
## تضخيم الأخطاء الاحتمالي (PEA)
أحد أبرز التحديات في ZNE هو تضخيم الضوضاء التي تؤثر على الدائرة المستهدفة بدقة. يوفر طي البوابات طريقة سهلة لإجراء هذا التضخيم، لكنه قد يكون غير دقيق وقد يؤدي إلى نتائج خاطئة. انظر مقالة ["Scalable error mitigation for noisy quantum circuits produces competitive expectation values"](https://arxiv.org/pdf/2108.09197)، وتحديدًا الصفحة 4 من المعلومات التكميلية للتفاصيل. يوفر تضخيم الأخطاء الاحتمالي نهجًا أكثر دقة لتضخيم الأخطاء من خلال تعلم الضوضاء.

PEA تقنية أكثر تطورًا تُجري تجارب أولية لإعادة بناء الضوضاء ثم تستخدم هذه المعلومات لإجراء تضخيم دقيق. تبدأ بتعلم نموذج الضوضاء الملوَّاة لكل طبقة من بوابات التشابك في الدائرة قبل تشغيلها (انظر [LayerNoiseLearningOptions](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-layer-noise-learning-options) لخيارات التعلم ذات الصلة). بعد مرحلة التعلم، تُنفَّذ الدوائر عند كل عامل ضوضاء، حيث تُضخَّم كل طبقة تشابك في الدوائر عن طريق حقن ضوضاء أحادية الـ Qubit بصورة احتمالية تتناسب مع نموذج الضوضاء المُتعلَّم المقابل. انظر مقالة ["Evidence for the utility of quantum computing before fault tolerance"](https://www.nature.com/articles/s41586-023-06096-3) لمزيد من التفاصيل.

يتكون PEA من ثلاث مراحل:
1. _التعلم_: يُتعلَّم نموذج الضوضاء الملوَّاة لكل طبقة من بوابات التشابك في الدائرة.
1. _تضخيم الضوضاء_: تُنفَّذ الدائرة الكمومية الأصلية عدة مرات عند عوامل ضوضاء مختلفة.
2. _الاستقراء_: يُقدَّر النتيجة المثالية عن طريق الاستقراء من نتائج قيم التوقع المشوشة نحو حد الضوضاء الصفرية.

بالنسبة للتجارب على نطاق الأداء، يكون PEA في الغالب الخيار الأفضل.

بما أن PEA هو تقنية لتضخيم ضوضاء ZNE، فأنت بحاجة أيضًا إلى تفعيل ZNE عن طريق ضبط `resilience.zne_mitigation = True`. يمكن استخدام خيارات [`resilience.zne`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-zne-options) الأخرى بالإضافة إلى ذلك لضبط المستقرِئات ومستويات التضخيم وما إلى ذلك. يتطلب PEA نموذج ضوضاء، والذي يُولَّد تلقائيًا عند استخدام الـ primitives.

يوفر المقتطف التالي مثالًا يُستخدَم فيه PEA لتخفيف نتيجة وظيفة Estimator:

In [6]:
estimator = Estimator(mode=backend)
estimator.options.resilience.zne_mitigation = True
estimator.options.resilience.zne.amplifier = "pea"

<span id="pec"></span>
## إلغاء الأخطاء الاحتمالي (PEC)
إلغاء الأخطاء الاحتمالي (PEC) هو تقنية لتخفيف الأخطاء في تقدير قيم التوقع للمراقِبات. وخلافًا لـ ZNE، يُعيد تقديرًا غير منحاز لقيمة التوقع. غير أنه يتكبد عمومًا عبئًا أكبر.

في PEC، يُعبَّر عن تأثير دائرة مستهدفة مثالية بوصفه تركيبًا خطيًا من الدوائر المشوشة القابلة للتنفيذ فعليًا في الواقع العملي:

$$
\mathcal{O}_{\text{ideal}} = \sum_{i} \eta_i \mathcal{O}_{noisy, i}
$$

يمكن حينئذٍ إعادة إنتاج مخرجات الدائرة المثالية عن طريق تنفيذ نسخ مختلفة من الدوائر المشوشة المُستخلَصة من مجموعة عشوائية محددة بالتركيب الخطي. إذا كانت المعاملات $\eta_i$ تُشكِّل توزيع احتمال، يمكن استخدامها مباشرةً بوصفها احتمالات المجموعة. في الواقع العملي، بعض المعاملات سالبة، لذا تُشكِّل توزيع شبه احتمالي بدلًا من ذلك. لا يزال بالإمكان استخدامها لتعريف مجموعة عشوائية، لكن ثمة عبء أخذ عينات مرتبط بسلبية توزيع الشبه الاحتمالي، والذي يتميز بالكمية

$$
\gamma = \sum_{i} \lvert \eta_i \rvert \geq 1.
$$

عبء أخذ العينات هو عامل ضربي على عدد اللقطات المطلوبة لتقدير قيمة التوقع بدقة معينة، مقارنةً بعدد اللقطات التي ستكون مطلوبة من الدائرة المثالية. يتناسب تربيعيًا مع $\gamma$، الذي بدوره يتناسب أسيًا مع عمق الدائرة.

يمكن تفعيل PEC عن طريق ضبط `pec_mitigation` على `True` في [خيارات مرونة Qiskit Runtime](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-resilience-options-v2) للـ Estimator.
خيارات Qiskit Runtime الخاصة بـ PEC موصوفة [هنا](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-pec-options). يمكن ضبط حد لعبء أخذ العينات باستخدام خيار `max_overhead`. لاحظ أن تحديد عبء أخذ العينات قد يتسبب في تجاوز دقة النتيجة للدقة المطلوبة. القيمة الافتراضية لـ `max_overhead` هي 100.

تُظهر خلية الكود التالية كيفية تفعيل PEC وضبط خيار `max_overhead` للـ Estimator.

In [7]:
estimator = Estimator(mode=backend)
estimator.options.resilience.pec_mitigation = True
estimator.options.resilience.pec.max_overhead = 100

## الخطوات التالية
- اطلع على [البرنامج التعليمي](/tutorials/combine-error-mitigation-techniques) حول دمج خيارات تخفيف الأخطاء مع الـ primitive الخاصة بـ Estimator.
- [ضبط تخفيف الأخطاء.](configure-error-mitigation)
- [ضبط قمع الأخطاء.](configure-error-suppression)
- استكشف [خيارات](runtime-options-overview) أخرى لـ primitives الخاصة بـ Qiskit Runtime.
- حدد [وضع التنفيذ](execution-modes) الذي تريد تشغيل وظيفتك فيه.